#  Notebook 11 — Interactive Dashboard (Plotly + Dash)

This notebook builds an interactive dashboard for exploring construction delay factors.

The dashboard includes:
- Interactive histograms  
- PCA scatter plots  
- Cluster visualizations  
- Delay Risk Index explorer  

The dashboard is built using Plotly and Dash, allowing users to interact with the data in real time.


In [62]:
!pip install plotly dash 

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [63]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, dcc, html
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans


In [64]:
df_scaled = pd.read_csv(
    r"C:\Users\olive\Desktop\My ML Material\CONSTRUCTION DELAY RISK PREDICTION SYSTEM\CDRPS\data\processed\df_scaled.csv"
)

df = df_scaled.copy()
df.head()


,Respondent Number,Category 1: Materials,Category 1: Materials.1,Category 1: Materials.2,Category 1: Materials.3,Category 1: Materials.4,Category 2: Labor and Equipment,Category 2: Labor and Equipment.1,Category 2: Labor and Equipment.2,Category 2: Labor and Equipment.3,...,Category 8: Scope of work.3,Category 9: External issues,Category 9: External issues .1,Category 9: External issues .2,Respondant Information,Respondant Information.1,Respondant Information.2,Respondant Information.3,Respondant Information.4,Respondant Information.5
0,-1.719981,1,1,1,1,1,1,1,1,1,...,1,2,2,3,1,2,2,2,3,5
1,-1.695756,2,5,3,2,3,2,2,2,3,...,3,1,1,4,1,3,2,3,3,3
2,-1.671530,1,2,1,1,2,2,1,2,1,...,2,1,1,4,1,2,2,1,2,4
3,-1.647305,2,2,3,3,2,2,3,4,4,...,3,4,3,3,1,3,1,1,3,4
4,-1.623080,1,2,1,1,2,2,1,2,1,...,2,1,1,4,1,2,3,1,2,4


In [65]:
numeric_df = df.select_dtypes(include=["int64", "float64"])

imputer = SimpleImputer(strategy="mean")
X_imputed = imputer.fit_transform(numeric_df)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

pca = PCA(n_components=2)
pca_components = pca.fit_transform(X_scaled)

df["PC1"] = pca_components[:, 0]
df["PC2"] = pca_components[:, 1]


In [66]:
kmeans = KMeans(n_clusters=4, random_state=42)
df["Cluster"] = kmeans.fit_predict(X_scaled)


In [67]:
df["Delay_Risk_Index"] = df.mean(axis=1)


BUILD DASH APP

In [68]:
#initialize the Dash app
app = Dash(__name__)

In [69]:
# CUSTOM CSS (Black + Red Theme)

custom_css = """
<style>

    /* GLOBAL BACKGROUND */
    body, .app-container, .dash-spreadsheet-container, .dash-table-container {
        background-color: black !important;
    }

    /* GLOBAL TEXT COLOR */
    h1, h2, h3, h4, h5, h6, p, span, div, label {
        color: red !important;
    }

    /* SIDEBAR */
    .sidebar {
        position: fixed;
        top: 0;
        left: 0;
        width: 230px;
        height: 100%;
        background-color: #111;
        padding: 20px;
        border-right: 2px solid red;
        box-shadow: 0 0 15px red;
    }

    .sidebar h2 {
        color: red;
        text-align: center;
        margin-bottom: 20px;
    }

    .sidebar a {
        display: block;
        padding: 10px;
        margin: 8px 0;
        color: red;
        text-decoration: none;
        border: 1px solid red;
        border-radius: 5px;
        transition: 0.3s;
    }

    .sidebar a:hover {
        background-color: red;
        color: black;
        box-shadow: 0 0 15px red;
    }

    /* MAIN CONTENT SHIFT */
    .main-content {
        margin-left: 260px;
        padding: 20px;
    }

    /* LOGO */
    .logo {
        width: 120px;
        height: auto;
        margin: 0 auto 20px auto;
        display: block;
        filter: drop-shadow(0 0 10px red);
    }

    /* GLOWING TABS */
    .tab {
        background-color: black !important;
        color: red !important;
        border: 1px solid red !important;
        transition: 0.3s;
    }

    .tab:hover {
        box-shadow: 0 0 10px red;
    }

    .tab--selected {
        background-color: white !important;
        color: black !important;
        border-bottom: 3px solid white !important;
        box-shadow: 0 0 15px white;
    }

    .tab {
        background-color: black !important;
        color: red !important;
        border: 1px solid red !important;
        transition: 0.3s;
    }

.tab:hover {
    box-shadow: 0 0 10px red;
}


    /* DROPDOWN */
    .Select-control, .Select-menu-outer, .Select-value-label {
        background-color: black !important;
        color: red !important;
        border: 1px solid red !important;
    }

    /* PLOTLY DARK THEME */
    .js-plotly-plot .plotly, .plot-container {
        background-color: black !important;
    }

    /* ANIMATIONS */
    @keyframes fadeIn {
        from {opacity: 0;}
        to {opacity: 1;}
    }

    .fade-in {
        animation: fadeIn 1.2s ease-in-out;
    }

</style>
"""

# DASH LAYOUT (FULLY INTEGRATED)

app.layout = html.Div([

    # Inject CSS
    html.Div(dcc.Markdown(custom_css), style={"display": "none"}),

    # SIDEBAR
    html.Div([
        html.Img(src="https://upload.wikimedia.org/wikipedia/commons/3/3f/Construction_icon.png",
                 className="logo"),

        html.H2("MENU"),

        html.A("Delay Factor Distributions", href="#tab1"),
        html.A("PCA Visualization", href="#tab2"),
        html.A("Cluster Profiles", href="#tab3"),
        html.A("Delay Risk Index", href="#tab4"),

    ], className="sidebar"),

    # MAIN CONTENT
    html.Div([

        html.H1("Construction Delay Analysis Dashboard",
                style={"textAlign": "center"},
                className="fade-in"),

        dcc.Tabs([

            # TAB 1
            dcc.Tab(label="Delay Factor Distributions", id="tab1", children=[
                html.H3("Select a Delay Factor", className="fade-in"),
                dcc.Dropdown(
                    id="dist-column",
                    options=[{"label": col, "value": col} for col in numeric_df.columns],
                    value=numeric_df.columns[0]
                ),
                dcc.Graph(id="dist-plot", className="fade-in")
            ]),

            # TAB 2
            dcc.Tab(label="PCA Visualization", id="tab2", children=[
                html.H3("PCA Scatter Plot", className="fade-in"),
                dcc.Graph(
                    id="pca-plot",
                    figure=px.scatter(
                        df, x="PC1", y="PC2", color="Cluster",
                        title="PCA Scatter Plot (PC1 vs PC2)",
                        hover_data=df.columns,
                        template="plotly_dark"
                    ),
                    className="fade-in"
                )
            ]),

            # TAB 3
            dcc.Tab(label="Cluster Profiles", id="tab3", children=[
                html.H3("Cluster Profile Heatmap", className="fade-in"),
                dcc.Graph(
                    id="cluster-heatmap",
                    figure=px.imshow(
                        df.groupby("Cluster").mean(),
                        aspect="auto",
                        color_continuous_scale="RdBu",
                        title="Cluster Profiles Heatmap",
                        template="plotly_dark"
                    ),
                    className="fade-in"
                )
            ]),

            # TAB 4
            dcc.Tab(label="Delay Risk Index", id="tab4", children=[
                html.H3("Delay Risk Index Distribution", className="fade-in"),
                dcc.Graph(
                    id="risk-dist",
                    figure=px.histogram(
                        df, x="Delay_Risk_Index", nbins=20, marginal="box",
                        title="Distribution of Delay Risk Index",
                        template="plotly_dark"
                    ),
                    className="fade-in"
                )
            ])
        ])

    ], className="main-content")

])


In [70]:
#Add callback for distribution plot
from dash.dependencies import Input, Output

@app.callback(
    Output("dist-plot", "figure"),
    Input("dist-column", "value")
)
def update_distribution(selected_column):
    fig = px.histogram(
        df, x=selected_column, nbins=20, marginal="box",
        title=f"Distribution of {selected_column}"
    )
    return fig


In [71]:
app.run(debug=False)